In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# PyTorch Imports
# ------------------------------------------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import torch.nn.functional as F

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ------------------------------------------------------------
# 1D-CNN Model Definition (PyTorch)
# ------------------------------------------------------------

class CNN1D(nn.Module):
    def __init__(self, input_dim, num_classes, cnn_params=None):
        super(CNN1D, self).__init__()
        
        # Default parameters
        default_params = {
            'filters': [64, 128, 64],
            'kernel_sizes': [3, 3, 3],
            'pool_sizes': [2, 2, 2],
            'dense_units': [128, 64],
            'dropout_rate': 0.3,
            'l2_reg': 0.001
        }
        
        if cnn_params:
            default_params.update(cnn_params)
        params = default_params
        
        self.params = params
        
        # Build convolutional layers
        self.conv_layers = nn.ModuleList()
        self.bn_layers = nn.ModuleList()
        self.pool_sizes = params['pool_sizes']
        
        in_channels = 1  # Start with 1 channel (features are in the length dimension)
        
        for i, (filters, kernel_size) in enumerate(zip(params['filters'], params['kernel_sizes'])):
            conv_layer = nn.Conv1d(
                in_channels=in_channels,
                out_channels=filters,
                kernel_size=kernel_size,
                padding=kernel_size // 2,  # Same padding
                bias=True
            )
            self.conv_layers.append(conv_layer)
            
            # Batch Normalization
            bn_layer = nn.BatchNorm1d(filters)
            self.bn_layers.append(bn_layer)
            
            in_channels = filters
        
        # Calculate flattened size after conv layers
        self.flattened_size = self._get_flattened_size(input_dim)
        
        # Build dense layers
        self.dense_layers = nn.ModuleList()
        self.dense_bn_layers = nn.ModuleList()
        
        prev_units = self.flattened_size
        for units in params['dense_units']:
            dense_layer = nn.Linear(prev_units, units)
            self.dense_layers.append(dense_layer)
            
            bn_layer = nn.BatchNorm1d(units)
            self.dense_bn_layers.append(bn_layer)
            
            prev_units = units
        
        # Output layer
        self.output_layer = nn.Linear(prev_units, num_classes)
        
        # Dropout
        self.dropout = nn.Dropout(params['dropout_rate'])
        
        # L2 regularization is handled by optimizer weight_decay
        
    def _get_flattened_size(self, input_dim):
        """Calculate the size after conv and pooling layers"""
        x = torch.randn(1, 1, input_dim)
        
        with torch.no_grad():
            for i, (conv, bn) in enumerate(zip(self.conv_layers, self.bn_layers)):
                x = conv(x)
                x = bn(x)
                x = F.relu(x)
                x = F.max_pool1d(x, kernel_size=self.pool_sizes[i])
            
            flattened = x.view(1, -1).size(1)
        
        return flattened
    
    def forward(self, x):
        # Convolutional layers
        for i, (conv, bn) in enumerate(zip(self.conv_layers, self.bn_layers)):
            x = conv(x)
            x = bn(x)
            x = F.relu(x)
            x = F.max_pool1d(x, kernel_size=self.pool_sizes[i])
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Dense layers
        for i, (dense, bn) in enumerate(zip(self.dense_layers, self.dense_bn_layers)):
            x = dense(x)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        
        # Output layer
        x = self.output_layer(x)
        
        return x

# ------------------------------------------------------------
# 2. 1D-CNN PIPELINE (PyTorch) - FIXED VERSION
# ------------------------------------------------------------

def run_1d_cnn_pytorch(df, 
                       target_column='label',
                       test_size=0.2,
                       random_state=42,
                       external_test_df=None,
                       preserve_order=True,
                       cnn_params=None,
                       batch_size=32,
                       epochs=100,
                       learning_rate=0.001,
                       patience=15):
    """
    1D-CNN Pipeline with PyTorch
    Supports both internal train-test split and external test set
    """
    
    print("="*80)
    print("1D-CNN PIPELINE (PyTorch)")
    print("="*80)
    print(f"Device: {device}")
    
    # Set random seeds for reproducibility
    torch.manual_seed(random_state)
    np.random.seed(random_state)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(random_state)
    
    # --------------------------------------------------------
    # 1. Data Preparation
    # --------------------------------------------------------
    
    df_processed = df.copy()
    
    if target_column not in df_processed.columns:
        available_cols = df_processed.columns.tolist()
        print(f"Warning: Target column '{target_column}' not found.")
        print(f"Available columns: {available_cols}")
        
        if 'label' in available_cols:
            target_column = 'label'
            print(f"Using 'label' as target column instead.")
        else:
            raise ValueError(f"Target column not found. Available: {available_cols}")
    
    all_features = [col for col in df_processed.columns if col != target_column]
    
    if not all_features:
        raise ValueError("No feature columns found!")
    
    print(f"\n[1] DATA PREPARATION")
    print(f"    Target column: {target_column}")
    print(f"    Number of features: {len(all_features)}")
    print(f"    Total samples: {len(df_processed)}")
    
    # Handle labels (convert to categorical if needed)
    le = LabelEncoder()
    df_processed['label_encoded'] = le.fit_transform(df_processed[target_column])
    
    label_info = df_processed[target_column].value_counts().sort_index()
    n_classes = len(label_info)
    print(f"    Number of classes: {n_classes}")
    print(f"    Class distribution:")
    for label, count in label_info.items():
        percentage = (count / len(df_processed)) * 100
        encoded_label = le.transform([label])[0]
        print(f"      Class {label} (encoded: {encoded_label}): {count} samples ({percentage:.1f}%)")
    
    # --------------------------------------------------------
    # 2. Train-Test Split (or use external test)
    # --------------------------------------------------------
    
    print(f"\n[2] DATA SPLITTING")
    
    if external_test_df is not None:
        # Use external test set
        print(f"    Using external test set provided")
        train_df = df_processed.copy()
        test_df = external_test_df.copy()
        
        # Encode test labels
        test_df['label_encoded'] = le.transform(test_df[target_column])
        
        # Ensure test set has same columns as train set
        missing_cols = set(train_df.columns) - set(test_df.columns)
        if missing_cols:
            print(f"    Warning: Test set missing columns: {missing_cols}")
            for col in missing_cols:
                if col not in [target_column, 'label_encoded']:
                    test_df[col] = 0
        
        # Keep only common columns
        common_cols = list(set(train_df.columns) & set(test_df.columns))
        train_df = train_df[common_cols]
        test_df = test_df[common_cols]
        
        print(f"    Training samples: {len(train_df)}")
        print(f"    Test samples: {len(test_df)}")
        
    else:
        # Internal train-test split
        if preserve_order:
            # Class-wise sequential split (preserves order AND label distribution)
            print(f"    Using CLASS-WISE sequential split (order preserved)")
            
            train_dfs = []
            test_dfs = []
            
            # Get unique encoded labels
            unique_labels = df_processed['label_encoded'].unique()
            print(f"    Splitting each of {len(unique_labels)} classes separately...")
            
            for label in unique_labels:
                # Get data for this class only
                class_data = df_processed[df_processed['label_encoded'] == label].copy()
                n_class_samples = len(class_data)
                
                if n_class_samples > 0:
                    # Calculate split index for this class
                    split_idx = int(n_class_samples * (1 - test_size))
                    
                    # Split this class's data
                    class_train = class_data.iloc[:split_idx]
                    class_test = class_data.iloc[split_idx:]
                    
                    train_dfs.append(class_train)
                    test_dfs.append(class_test)
                    
                    original_label = le.inverse_transform([label])[0]
                    print(f"      Class {original_label} (enc: {label}): {n_class_samples} total -> "
                          f"{len(class_train)} train, {len(class_test)} test")
            
            # Combine all classes
            train_df = pd.concat(train_dfs, ignore_index=False).sort_index()
            test_df = pd.concat(test_dfs, ignore_index=False).sort_index()
            
            print(f"\n    Training samples: {len(train_df)}")
            print(f"    Test samples: {len(test_df)}")
            
        else:
            # Random split with stratification
            print(f"    Using random split with stratification")
            if n_classes > 1:
                train_df, test_df = train_test_split(
                    df_processed, 
                    test_size=test_size, 
                    stratify=df_processed['label_encoded'], 
                    random_state=random_state
                )
            else:
                print("    Warning: Only one class found. Stratification disabled.")
                train_df, test_df = train_test_split(
                    df_processed, 
                    test_size=test_size, 
                    random_state=random_state
                )
            print(f"    Training samples: {len(train_df)}")
            print(f"    Test samples: {len(test_df)}")
    
    # --------------------------------------------------------
    # 3. Data Preparation for PyTorch
    # --------------------------------------------------------
    
    print(f"\n[3] DATA PREPARATION FOR PyTorch")
    
    # Separate features and target
    feature_cols = [col for col in train_df.columns if col not in [target_column, 'label_encoded']]
    
    X_train = train_df[feature_cols].values.astype(np.float32)
    y_train = train_df['label_encoded'].values.astype(np.int64)
    X_test = test_df[feature_cols].values.astype(np.float32)
    y_test = test_df['label_encoded'].values.astype(np.int64)
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Reshape for 1D-CNN: (samples, 1, features)
    X_train_reshaped = X_train_scaled.reshape(X_train_scaled.shape[0], 1, X_train_scaled.shape[1])
    X_test_reshaped = X_test_scaled.reshape(X_test_scaled.shape[0], 1, X_test_scaled.shape[1])
    
    print(f"    Features scaled using StandardScaler")
    print(f"    Reshaped for 1D-CNN: {X_train_reshaped.shape}")
    
    # --------------------------------------------------------
    # 4. Create PyTorch Datasets and DataLoaders
    # --------------------------------------------------------
    
    print(f"\n[4] CREATING PyTorch DATALOADERS")
    
    # Convert to PyTorch tensors
    X_train_tensor = torch.FloatTensor(X_train_reshaped)
    y_train_tensor = torch.LongTensor(y_train)
    X_test_tensor = torch.FloatTensor(X_test_reshaped)
    y_test_tensor = torch.LongTensor(y_test)
    
    # Create datasets
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    
    # Split training data for validation
    val_size = int(0.1 * len(train_dataset))
    train_size = len(train_dataset) - val_size
    
    train_subset, val_subset = random_split(
        train_dataset, 
        [train_size, val_size],
        generator=torch.Generator().manual_seed(random_state)
    )
    
    # Create dataloaders
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    print(f"    Train samples: {len(train_subset)}")
    print(f"    Validation samples: {len(val_subset)}")
    print(f"    Test samples: {len(test_dataset)}")
    print(f"    Batch size: {batch_size}")
    
    # --------------------------------------------------------
    # 5. Build Model
    # --------------------------------------------------------
    
    print(f"\n[5] BUILDING 1D-CNN MODEL")
    
    # Initialize model
    model = CNN1D(
        input_dim=X_train_reshaped.shape[2],
        num_classes=n_classes,
        cnn_params=cnn_params
    ).to(device)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"\n    Model Architecture:")
    print(f"    Total parameters: {total_params:,}")
    print(f"    Trainable parameters: {trainable_params:,}")
    
    # Loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=0.001)
    
    # Learning rate scheduler - FIXED: removed verbose argument
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )
    
    # --------------------------------------------------------
    # 6. Training Loop
    # --------------------------------------------------------
    
    print(f"\n[6] TRAINING 1D-CNN MODEL")
    print(f"    Max epochs: {epochs}")
    print(f"    Early stopping patience: {patience}")
    print(f"    Initial learning rate: {learning_rate}")
    
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Statistics
            train_loss += loss.item() * batch_X.size(0)
            _, predicted = torch.max(outputs, 1)
            train_total += batch_y.size(0)
            train_correct += (predicted == batch_y).sum().item()
        
        train_loss = train_loss / len(train_subset)
        train_acc = train_correct / train_total
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                
                val_loss += loss.item() * batch_X.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += batch_y.size(0)
                val_correct += (predicted == batch_y).sum().item()
        
        val_loss = val_loss / len(val_subset)
        val_acc = val_correct / val_total
        
        # Update learning rate
        scheduler.step(val_loss)
        
        # Save metrics
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        
        # Print progress
        if (epoch + 1) % 10 == 0 or epoch == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"    Epoch [{epoch+1}/{epochs}] - "
                  f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
                  f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, "
                  f"LR: {current_lr:.6f}")
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n    Early stopping triggered at epoch {epoch+1}")
                break
    
    # Load best model
    if best_model_state:
        model.load_state_dict(best_model_state)
        print(f"\n    Loaded best model with validation loss: {best_val_loss:.4f}")
    
    print(f"\n    Training completed!")
    
    # --------------------------------------------------------
    # 7. Model Evaluation
    # --------------------------------------------------------
    
    print(f"\n[7] MODEL EVALUATION")
    
    # Evaluation function
    def evaluate(loader, dataset):
        model.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for batch_X, batch_y in loader:
                batch_X = batch_X.to(device)
                outputs = model(batch_X)
                _, predicted = torch.max(outputs, 1)
                
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(batch_y.numpy())
        
        return np.array(all_preds), np.array(all_labels)
    
    # Get predictions
    y_train_pred, y_train_true = evaluate(train_loader, train_subset)
    y_test_pred, y_test_true = evaluate(test_loader, test_dataset)
    
    # Training results
    train_results = {
        'accuracy': accuracy_score(y_train_true, y_train_pred),
        'f1_weighted': f1_score(y_train_true, y_train_pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_train_true, y_train_pred, average='weighted', zero_division=0),
        'precision_weighted': precision_score(y_train_true, y_train_pred, average='weighted', zero_division=0),
        'n_samples': len(y_train_true)
    }
    
    # Test results
    test_results = {
        'accuracy': accuracy_score(y_test_true, y_test_pred),
        'f1_weighted': f1_score(y_test_true, y_test_pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_test_true, y_test_pred, average='weighted', zero_division=0),
        'precision_weighted': precision_score(y_test_true, y_test_pred, average='weighted', zero_division=0),
        'confusion_matrix': confusion_matrix(y_test_true, y_test_pred),
        'classification_report': classification_report(y_test_true, y_test_pred, digits=3),
        'n_samples': len(y_test_true)
    }
    
    # Training history
    train_history = {
        'train_loss': train_losses,
        'val_loss': val_losses,
        'train_accuracy': train_accs,
        'val_accuracy': val_accs
    }
    
    # --------------------------------------------------------
    # 8. Display Results
    # --------------------------------------------------------
    
    print("\n" + "="*80)
    print("1D-CNN RESULTS (PyTorch)")
    print("="*80)
    
    print(f"\nDataset Information:")
    print(f"  Total samples: {len(df_processed)}")
    print(f"  Number of classes: {n_classes}")
    print(f"  Target column: '{target_column}'")
    print(f"  Number of features: {len(all_features)}")
    print(f"  Input shape for CNN: (1, {X_train_reshaped.shape[2]})")
    
    print(f"\nData Split:")
    print(f"  Training samples: {len(train_subset)}")
    print(f"  Validation samples: {len(val_subset)}")
    print(f"  Test samples: {len(test_dataset)}")
    
    print(f"\nTraining Set Performance:")
    print(f"  Accuracy: {train_results['accuracy']:.4f}")
    print(f"  F1 Weighted: {train_results['f1_weighted']:.4f}")
    print(f"  Recall Weighted: {train_results['recall_weighted']:.4f}")
    print(f"  Precision Weighted: {train_results['precision_weighted']:.4f}")
    
    print(f"\nTest Set Performance:")
    print(f"  Accuracy: {test_results['accuracy']:.4f}")
    print(f"  F1 Weighted: {test_results['f1_weighted']:.4f}")
    print(f"  Recall Weighted: {test_results['recall_weighted']:.4f}")
    print(f"  Precision Weighted: {test_results['precision_weighted']:.4f}")
    
    print(f"\nBest Validation Accuracy: {max(val_accs):.4f}")
    print(f"Best Validation Loss: {min(val_losses):.4f}")
    
    print(f"\nConfusion Matrix (Test Set):")
    print(test_results['confusion_matrix'])
    
    print(f"\nClassification Report (Test Set):")
    print(test_results['classification_report'])
    
    print("\n" + "="*80)
    print("1D-CNN PIPELINE (PyTorch) COMPLETED")
    print("="*80)
    
    # Return all results
    return {
        'train_df': train_df,
        'test_df': test_df,
        'model': model,
        'scaler': scaler,
        'label_encoder': le,
        'train_results': train_results,
        'test_results': test_results,
        'train_history': train_history,
        'all_features': all_features,
        'n_classes': n_classes,
        'target_column': target_column,
        'preserve_order': preserve_order,
        'cnn_params': model.params if hasattr(model, 'params') else cnn_params,
        'device': device
    }

# ------------------------------------------------------------
# 9. MODE FUNCTIONS (Same interface as before)
# ------------------------------------------------------------

def run_cnn_mode_1_single_file_pytorch(file_path, preserve_order=False, **kwargs):
    """
    MODE 1: Read a single CSV file and split into train/test internally
    """
    print("="*80)
    print("MODE 1: SINGLE FILE TRAIN/TEST SPLIT (PyTorch)")
    print("="*80)
    print(f"Note: Order preservation = {preserve_order}")
    if preserve_order:
        print("      Using CLASS-WISE sequential split")
    
    df = pd.read_csv(file_path)
    print(f"✓ Loaded data from: {file_path}")
    print(f"✓ Data shape: {df.shape}")
    print(f"✓ Columns: {df.columns.tolist()}")
    
    # Ensure preserve_order is passed to pipeline
    kwargs['preserve_order'] = preserve_order
    
    results = run_1d_cnn_pytorch(df=df, **kwargs)
    return results

def run_cnn_mode_2_two_files_pytorch(train_file_path, test_file_path, **kwargs):
    """
    MODE 2: Read two separate CSV files for train and test
    """
    print("="*80)
    print("MODE 2: SEPARATE TRAIN/TEST FILES (PyTorch)")
    print("="*80)
    print("Note: Order is preserved from original files")
    
    # Load data
    train_df = pd.read_csv(train_file_path)
    test_df = pd.read_csv(test_file_path)
    
    print(f"✓ Train file: {train_file_path}, Shape: {train_df.shape}")
    print(f"✓ Test file: {test_file_path}, Shape: {test_df.shape}")
    
    # Ensure both have the same columns
    train_cols = set(train_df.columns)
    test_cols = set(test_df.columns)
    
    if train_cols != test_cols:
        print("⚠ Warning: Train and test files have different columns!")
        print(f"  Train columns: {train_df.columns.tolist()}")
        print(f"  Test columns: {test_df.columns.tolist()}")
        
        # Keep only common columns
        common_cols = list(train_cols.intersection(test_cols))
        print(f"✓ Using common columns: {common_cols}")
        
        if len(common_cols) < 2:
            raise ValueError("✗ Not enough common columns between train and test files!")
        
        train_df = train_df[common_cols]
        test_df = test_df[common_cols]
    
    # Run pipeline with external test set
    results = run_1d_cnn_pytorch(
        df=train_df,
        external_test_df=test_df,
        preserve_order=False,  # Always preserve order for external test
        **kwargs
    )
    return results

def run_cnn_mode_3_dataframes_pytorch(train_df, test_df=None, preserve_order=True, **kwargs):
    """
    MODE 3: Use existing DataFrames
    """
    if test_df is None:
        print("="*80)
        print("MODE 3: SINGLE DATAFRAME (PyTorch)")
        print("="*80)
        print(f"Note: Order preservation = {preserve_order}")
        if preserve_order:
            print("      Using CLASS-WISE sequential split")
        
        kwargs['preserve_order'] = preserve_order
        results = run_1d_cnn_pytorch(df=train_df, **kwargs)
    else:
        print("="*80)
        print("MODE 3: TWO DATAFRAMES (PyTorch)")
        print("="*80)
        print("Note: Order is preserved from input DataFrames")
        
        results = run_1d_cnn_pytorch(
            df=train_df,
            external_test_df=test_df,
            preserve_order=False,
            **kwargs
        )
    return results

# ------------------------------------------------------------
# 10. EXAMPLE USAGE
# ------------------------------------------------------------

if __name__ == "__main__":
    """
    Example usage of the 1D-CNN pipeline with PyTorch
    """
    
    print("\n" + "="*80)
    print("1D-CNN PIPELINE WITH PyTorch - EXAMPLE USAGE")
    print("="*80)
    
    # Generate sample data
    np.random.seed(42)
    n_samples = 1000
    n_features = 50
    
    sample_data = pd.DataFrame(
        np.random.randn(n_samples, n_features),
        columns=[f'feature_{i}' for i in range(n_features)]
    )
    sample_data['label'] = np.random.choice(['A', 'B', 'C'], n_samples)
    
    # Save sample data
    sample_data.to_csv('sample_data.csv', index=False)
    
    print("\n✓ Created sample data with:")
    print(f"  - {n_samples} samples")
    print(f"  - {n_features} features")
    print(f"  - 3 classes (A, B, C)")
    
    # Example 1: Single file mode
    print("\n" + "-"*60)
    print("EXAMPLE 1: Single file with preserve_order=True")
    print("-"*60)
    
    results1 = run_cnn_mode_1_single_file_pytorch(
        'sample_data.csv',
        target_column='label',
        preserve_order=True,
        test_size=0.2,
        batch_size=32,
        epochs=50,  # Reduced for example
        learning_rate=0.001,
        cnn_params={
            'filters': [32, 64, 32],
            'kernel_sizes': [3, 3, 3],
            'pool_sizes': [2, 2, 2],
            'dense_units': [64, 32],
            'dropout_rate': 0.2
        }
    )
    
    print(f"\n✓ Test Accuracy: {results1['test_results']['accuracy']:.4f}")
    
    # Example 2: Two files mode
    print("\n" + "-"*60)
    print("EXAMPLE 2: Two files mode")
    print("-"*60)
    
    # Create train/test splits
    train_data = sample_data.iloc[:700]
    test_data = sample_data.iloc[700:]
    
    train_data.to_csv('train_data.csv', index=False)
    test_data.to_csv('test_data.csv', index=False)
    
    results2 = run_cnn_mode_2_two_files_pytorch(
        'train_data.csv',
        'test_data.csv',
        target_column='label',
        batch_size=32,
        epochs=50,
        cnn_params={
            'filters': [32, 64],
            'kernel_sizes': [3, 3],
            'pool_sizes': [2, 2],
            'dense_units': [64],
            'dropout_rate': 0.3
        }
    )
    
    print(f"\n✓ Test Accuracy: {results2['test_results']['accuracy']:.4f}")
    
    # Clean up
    import os
    os.remove('sample_data.csv')
    os.remove('train_data.csv')
    os.remove('test_data.csv')
    print("\n✓ Cleaned up sample files")

Using device: cuda

1D-CNN PIPELINE WITH PyTorch - EXAMPLE USAGE

✓ Created sample data with:
  - 1000 samples
  - 50 features
  - 3 classes (A, B, C)

------------------------------------------------------------
EXAMPLE 1: Single file with preserve_order=True
------------------------------------------------------------
MODE 1: SINGLE FILE TRAIN/TEST SPLIT (PyTorch)
Note: Order preservation = True
      Using CLASS-WISE sequential split
✓ Loaded data from: sample_data.csv
✓ Data shape: (1000, 51)
✓ Columns: ['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22', 'feature_23', 'feature_24', 'feature_25', 'feature_26', 'feature_27', 'feature_28', 'feature_29', 'feature_30', 'feature_31', 'feature_32', 'feature_33', 'feature_34', 'featu

In [6]:
results = run_cnn_mode_2_two_files_pytorch(
    train_file_path='DATASETS/CDR-MLC/level_1.csv',
    test_file_path='DATASETS/CDR-MLC/level_3.csv',
    )

MODE 2: SEPARATE TRAIN/TEST FILES (PyTorch)
Note: Order is preserved from original files
✓ Train file: DATASETS/CDR-MLC/level_1.csv, Shape: (68079, 37)
✓ Test file: DATASETS/CDR-MLC/level_3.csv, Shape: (60482, 37)
1D-CNN PIPELINE (PyTorch)
Device: cuda

[1] DATA PREPARATION
    Target column: label
    Number of features: 36
    Total samples: 68079
    Number of classes: 5
    Class distribution:
      Class HTTP (encoded: 0): 39798 samples (58.5%)
      Class SFTP (encoded: 1): 11422 samples (16.8%)
      Class SMTP (encoded: 2): 6267 samples (9.2%)
      Class SSH (encoded: 3): 3127 samples (4.6%)
      Class VIDEO (encoded: 4): 7465 samples (11.0%)

[2] DATA SPLITTING
    Using external test set provided
    Training samples: 68079
    Test samples: 60482

[3] DATA PREPARATION FOR PyTorch
    Features scaled using StandardScaler
    Reshaped for 1D-CNN: (68079, 1, 36)

[4] CREATING PyTorch DATALOADERS
    Train samples: 61272
    Validation samples: 6807
    Test samples: 60482
   

In [7]:
results = run_cnn_mode_2_two_files_pytorch(
    train_file_path='DATASETS/CDR-MLC/level_1.csv',
    test_file_path='DATASETS/CDR-MLC/level_2.csv',
    )

MODE 2: SEPARATE TRAIN/TEST FILES (PyTorch)
Note: Order is preserved from original files
✓ Train file: DATASETS/CDR-MLC/level_1.csv, Shape: (68079, 37)
✓ Test file: DATASETS/CDR-MLC/level_2.csv, Shape: (201165, 37)
1D-CNN PIPELINE (PyTorch)
Device: cuda

[1] DATA PREPARATION
    Target column: label
    Number of features: 36
    Total samples: 68079
    Number of classes: 5
    Class distribution:
      Class HTTP (encoded: 0): 39798 samples (58.5%)
      Class SFTP (encoded: 1): 11422 samples (16.8%)
      Class SMTP (encoded: 2): 6267 samples (9.2%)
      Class SSH (encoded: 3): 3127 samples (4.6%)
      Class VIDEO (encoded: 4): 7465 samples (11.0%)

[2] DATA SPLITTING
    Using external test set provided
    Training samples: 68079
    Test samples: 201165

[3] DATA PREPARATION FOR PyTorch
    Features scaled using StandardScaler
    Reshaped for 1D-CNN: (68079, 1, 36)

[4] CREATING PyTorch DATALOADERS
    Train samples: 61272
    Validation samples: 6807
    Test samples: 201165
